In [1]:
import os 
import sys
import pandas as pd
import s3fs
import duckdb
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

In [2]:
fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY"),
    client_kwargs={"region_name": os.getenv("AWS_REGION")},
)

bucket = os.getenv("S3_BUCKET_RAW")

with fs.open(f"s3://{bucket}/fastf1/2026/1/laps.parquet", "rb") as f:
    df_laps = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/1/results.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/1/telemetry.parquet", "rb") as f:
    df_telemetry = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/1/weather.parquet", "rb") as f:
    df_weather = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/2/laps.parquet", "rb") as f:
    df_laps2 = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/2/results.parquet", "rb") as f:
    df_results2 = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/2/telemetry.parquet", "rb") as f:
    df_telemetry2 = pd.read_parquet(f)
with fs.open(f"s3://{bucket}/fastf1/2026/2/weather.parquet", "rb") as f:
    df_weather2 = pd.read_parquet(f)

print(df_laps.shape)
print(df_results.shape)
print(df_telemetry.shape)
print(df_weather.shape)
print(df_laps2.shape)
print(df_results2.shape)
print(df_telemetry2.shape)
print(df_weather2.shape)

(1007, 35)
(22, 26)
(684078, 24)
(148, 12)
(924, 35)
(22, 26)
(706701, 24)
(152, 12)


In [3]:
from ingestion.fastf1.bronze_loader import load_fastf1_bronze, verify_bronze_tables
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))
load_fastf1_bronze(con, year=2026)
verify_bronze_tables(con)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

bronze.fastf1_laps                       rows=1931   rounds=2
bronze.fastf1_results                    rows=44     rounds=2
bronze.fastf1_telemetry                  rows=1390779 rounds=2
bronze.fastf1_weather                    rows=300    rounds=2


In [6]:
result = con.execute("""
    SELECT driver_id, team_name, grid_position, position, points
    FROM bronze.fastf1_results
    WHERE season = 2026 AND round = 1
    ORDER BY position
    LIMIT 5
""").df()

print("2026 Australian Top 5:")
print(result.to_string(index=False))

2026 Australian Top 5:
driver_id team_name  grid_position  position  points
  russell  Mercedes            1.0       1.0    25.0
antonelli  Mercedes            2.0       2.0    18.0
  leclerc   Ferrari            4.0       3.0    15.0
 hamilton   Ferrari            7.0       4.0    12.0
   norris   McLaren            6.0       5.0    10.0


In [8]:
lap_check = con.execute("""
    SELECT
        round,
        COUNT(*) AS total_laps,
        COUNT(DISTINCT driver) AS drivers,
        ROUND(MIN(lap_time), 3) AS fastest_lap,
        ROUND(MAX(lap_time), 3) AS slowest_lap
    FROM bronze.fastf1_laps
    WHERE season = 2026
    AND is_accurate = true
    GROUP BY round
    ORDER BY round
""").df()

print(lap_check.to_string(index=False))

 round  total_laps  drivers  fastest_lap  slowest_lap
     1         835       20  82091000000  96209000000
     2         816       18  95275000000 110154000000


In [9]:
tel_check = con.execute("""
    SELECT
        round,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT driver) AS drivers,
        COUNT(DISTINCT lap_number) AS laps
    FROM bronze.fastf1_telemetry
    WHERE season = 2026
    GROUP BY round
    ORDER BY round
""").df()

print(tel_check.to_string(index=False))

 round  total_rows  drivers  laps
     1      684078       21    58
     2      706701       22    56
